# 文本特征向量化：词袋模型、Word2Vec 以及 TF-IDF 介绍

## 1. 学习目标

通过本章的学习，读者将能够：

1. 理解文本特征向量化在自然语言处理（NLP）中的重要性。
2. 掌握 **词袋模型（BoW）** 的构建过程及其局限性。
3. 掌握 **TF-IDF** 的计算公式、物理含义及其在关键词提取中的应用。
4. 理解 **Word2Vec** 的基本思想（CBOW 与 Skip-Gram）及其相对于传统方法的优势。
5. 能够使用 Python 工具库（`sklearn`、`gensim`）实现上述文本向量化方法。

---

## 2. 引言

在自然语言处理（NLP）领域，计算机无法直接理解人类语言的文本信息（如单词、句子）。为了让计算机能够处理这些信息，我们需要将文本转换为数值向量，这一过程被称为**文本特征向量化**（Text Vectorization）。

文本向量化是 NLP 任务（如文本分类、情感分析、机器翻译）的基石。高质量的向量表示能够更好地捕捉文本的语义信息，从而提升下游模型的性能。常见的文本向量化方法经历了从基于统计的**词袋模型（BoW）**、**TF-IDF**，到基于神经网络的 **Word2Vec** 的演变。

---


## 3. 文本预处理（Text Preprocessing）

在进行特征向量化之前，通常需要对原始文本进行清洗和标准化。这一步骤对于降低噪音、减小特征维度至关重要。

### 3.1 核心步骤

1. **分词（Tokenization）**：将连续的文本序列切割成单词（Token）或词组。
2. **去除停用词（Stop Words Removal）**：过滤掉像 "the", "is", "at", "which" 这样出现频率极高但对文本语义贡献很小的词。
3. **词形标准化（Normalization）**：包括词干提取（Stemming）和词形还原（Lemmatization）。

### 3.2 Python 实现示例


In [6]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string

# 下载必要的数据包 (首次运行需要)
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
    nltk.data.find('corpora/wordnet')
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')
    nltk.download('wordnet')
    nltk.download('omw-1.4')
    nltk.download('punkt_tab')

def preprocess_text(text):
    # 1. 分词
    tokens = nltk.word_tokenize(text.lower())
    
    # 2. 去除停用词和标点符号
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words and token not in string.punctuation]
    
    # 3. 词形还原
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    return tokens

# 示例
text = "The quick brown foxes are jumping over the lazy dog."
processed_tokens = preprocess_text(text)
print(f"原始文本: {text}")
print(f"预处理后: {processed_tokens}")

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/jovyan/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


原始文本: The quick brown foxes are jumping over the lazy dog.
预处理后: ['quick', 'brown', 'fox', 'jumping', 'lazy', 'dog']


## 4. 词袋模型（Bag of Words, BoW）

### 4.1 词袋模型的基本原理

**词袋模型（Bag of Words, BoW）** 是最基础的文本表示方法。它将文本看作是一个装满词汇的袋子，忽略词语在文本中出现的顺序和语法结构，仅关注每个词出现的**频次**。

#### 4.1.1 构建步骤

1. **构建词汇表（Vocabulary）**：扫描所有文档，统计出所有出现过的不同单词，构成一个有序的词汇表。
2. **向量化（Vectorization）**：对于每一个文档，根据词汇表构建一个向量。向量的每一维对应词汇表中的一个词，数值表示该词在文档中出现的次数。

#### 4.1.2 手动计算示例

假设我们有以下两个简单的文档：

- 文档 A：`I love NLP`
- 文档 B：`I love coding`

**第一步：构建词汇表**：

所有出现的单词为：`I`, `love`, `NLP`, `coding`。
排序后的词汇表为：`['coding', 'I', 'love', 'NLP']`。

**第二步：生成向量**：

- **文档 A** (`I love NLP`):

  - `coding`: 0 次
  - `I`: 1 次
  - `love`: 1 次
  - `NLP`: 1 次
  - 向量 A = `[0, 1, 1, 1]`

- **文档 B** (`I love coding`):
  - `coding`: 1 次
  - `I`: 1 次
  - `love`: 1 次
  - `NLP`: 0 次
  - 向量 B = `[1, 1, 1, 0]`

### 4.2 词袋模型的特点

**优势：**

- **简单直观**：易于理解和实现。
- **计算高效**：对于小规模数据集，构建速度快。
- **适用性**：在简单的文本分类（如垃圾邮件识别）任务中表现尚可。

**局限性：**

- **忽略语序**：无法区分 "Not bad, good" 和 "Not good, bad" 的语义区别。
- **数据稀疏**：当词汇表很大（如 10 万词）而单个文档很短时，向量中绝大多数元素为 0，造成内存浪费和计算困难。
- **缺乏语义**：无法捕捉词与词之间的相似性（例如 "car" 和 "automobile" 被视为完全无关的两个维度）。

### 4.3 词袋模型 Python 实现

使用 `sklearn.feature_extraction.text.CountVectorizer` 可以快速实现词袋模型。

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

# 示例文档
documents = [
    "I love machine learning",
    "Machine learning is amazing",
    "Deep learning builds on machine learning"
]

# 初始化 CountVectorizer
# stop_words='english' 表示移除英语停用词（如 is, the, on 等无实际意义的词）
vectorizer = CountVectorizer(stop_words='english')

# 拟合数据并转换为矩阵
X = vectorizer.fit_transform(documents)

# 输出结果
print("词汇表 (Feature Names):", vectorizer.get_feature_names_out())
print("\n词频矩阵 (Dense Representation):\n", X.toarray())

词汇表 (Feature Names): ['amazing' 'builds' 'deep' 'learning' 'love' 'machine']

词频矩阵 (Dense Representation):
 [[0 0 0 1 1 1]
 [1 0 0 1 0 1]
 [0 1 1 2 0 1]]


**输出结果：**

```text
词汇表 (Feature Names): ['amazing' 'builds' 'deep' 'learning' 'love' 'machine']

词频矩阵 (Dense Representation):
 [[0 0 0 1 1 1]
 [1 0 0 1 0 1]
 [0 1 1 2 0 1]]
```

---

## 5. TF-IDF：基于统计的文本表示

### 5.1 TF-IDF 的核心思想

在词袋模型中，单纯使用词频（TF）存在一个问题：某些常见词（如 "is", "that", "program"）可能在所有文档中都频繁出现，但它们对区分文档主题的贡献很小。相反，一些罕见词（如 "neural", "gradient"）更能代表文档的特性。

**TF-IDF（Term Frequency-Inverse Document Frequency）** 旨在解决这一问题。它通过降低常见词的权重，同时提升罕见词的权重，来评估一个词对一个文件集或一个语料库中的其中一份文件的重要程度。

### 5.2 计算公式详解

TF-IDF 是 **TF（词频）** 和 **IDF（逆文档频率）** 的乘积：

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)
$$

其中 $t$ 代表单词，$d$ 代表文档。

#### 5.2.1 词频 (Term Frequency, TF)

表示单词 $t$ 在文档 $d$ 中出现的次数（或频率）。

$$
\text{TF}(t, d) = \frac{\text{单词 } t \text{ 在文档 } d \text{ 中出现的次数}}{\text{文档 } d \text{ 的总词数}}
$$

> **注**：`sklearn` 等库中通常直接使用原始计数作为 TF。

#### 5.2.2 逆文档频率 (Inverse Document Frequency, IDF)

衡量单词 $t$ 的稀有程度。如果一个词在越少的文档中出现，它的 IDF 值就越高。

$$
\text{IDF}(t) = \log \left( \frac{N}{\text{DF}(t) + 1} \right) + 1
$$

- $N$：语料库中的文档总数。
- $\text{DF}(t)$：包含单词 $t$ 的文档数量。
- **为什么要加 1？** 分母加 1 是为了平滑处理（Smoothing），防止 $\text{DF}(t)=0$ 时出现除零错误。
- **为什么要取对数？** 对数函数可以抑制数值过大，同时使得权重变化更加平滑，符合信息论中信息量的定义。

### 5.3 TF-IDF 的特点

**优势：**

- **过滤噪音**：自动降低常见词（高频但无实际意义）的权重。
- **突出重点**：能较好地提取文档的关键词。

**局限性：**

- **仍基于统计**：依然忽略了词序和上下文语义。
- **数据稀疏**：生成的矩阵依然是高维稀疏矩阵。

### 5.4 TF-IDF Python 实现

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    "I love machine learning",
    "Machine learning is amazing",
    "Deep learning builds on machine learning"
]

# 初始化 TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english')

# 转换
X = vectorizer.fit_transform(documents)

print("词汇表:", vectorizer.get_feature_names_out())
print("\nTF-IDF 矩阵:\n", X.toarray())

词汇表: ['amazing' 'builds' 'deep' 'learning' 'love' 'machine']

TF-IDF 矩阵:
 [[0.         0.         0.         0.45329466 0.76749457 0.45329466]
 [0.76749457 0.         0.         0.45329466 0.         0.45329466]
 [0.         0.51680194 0.51680194 0.61046311 0.         0.30523155]]


**输出结果分析：**

```text
词汇表: ['amazing' 'builds' 'deep' 'learning' 'love' 'machine']

TF-IDF 矩阵:
 [[0.         0.         0.         0.45329466 0.76749457 0.45329466]
 [0.76749457 0.         0.         0.45329466 0.         0.45329466]
 [0.         0.51680194 0.51680194 0.61046311 0.         0.30523155]]
```

可以看到，在第一个文档中，"love" 的权重（0.767）高于 "machine"（0.453）和 "learning"（0.453），因为 "love" 只出现在这一个文档中（更稀有），而 "machine" 和 "learning" 在多个文档中出现。

---

## 6. Word2Vec：基于神经网络的词向量表示

### 6.1 Word2Vec 的核心思想

无论是 BoW 还是 TF-IDF，它们都属于**离散表示**（Discrete Representation），词与词之间是独立的。**Word2Vec** 提出了一种**分布式表示**（Distributed Representation）方法，将每个词映射到一个低维、稠密的实数向量空间中。

Word2Vec 的核心假设是：**具有相似上下文的词，其语义也是相似的**（Distributional Hypothesis）。

### 6.2 两种训练架构：CBOW 与 Skip-Gram

Word2Vec 通过浅层神经网络进行训练，主要包含两种架构：

1. **CBOW (Continuous Bag of Words)**：

   - **任务**：根据上下文单词预测中心单词。
   - **场景**：例如句子 "The cat sits on the mat"，CBOW 会尝试用 `["The", "cat", "on", "the", "mat"]` 来预测 `sits`。
   - **特点**：训练速度快，适合经常出现的词。

2. **Skip-Gram**：
   - **任务**：根据中心单词预测上下文单词。
   - **场景**：用 `sits` 来预测 `["The", "cat", "on", "the", "mat"]`。
   - **特点**：对罕见词和生僻词的效果更好，需要更多的训练数据。

### 6.3 语义特性：向量运算

Word2Vec 训练出的向量具有惊人的语义特性，支持向量代数运算：

$$
\text{Vector(King)} - \text{Vector(Man)} + \text{Vector(Woman)} \approx \text{Vector(Queen)}
$$

这意味着模型“学会”了性别这一语义概念。

### 6.4 Word2Vec Python 实现

使用 `gensim` 库可以轻松训练 Word2Vec 模型。

In [9]:
from gensim.models import Word2Vec

# 准备语料（分词后的列表）
sentences = [
    ["I", "love", "machine", "learning"],
    ["Machine", "learning", "is", "amazing"],
    ["Deep", "learning", "builds", "on", "machine", "learning"]
]

# 训练模型
# vector_size: 词向量维度（通常设为 100-300）
# window: 上下文窗口大小
# min_count: 忽略出现次数少于该值的词
# workers: 训练使用的线程数
model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

# 获取单词 'machine' 的向量
vector = model.wv["machine"]
print(f"'machine' 的向量维度: {vector.shape}")
print(f"'machine' 的前 10 维数值: {vector[:10]}")

# 计算相似度
similarity = model.wv.similarity("machine", "learning")
print(f"\n'machine' 和 'learning' 的余弦相似度: {similarity:.4f}")

'machine' 的向量维度: (100,)
'machine' 的前 10 维数值: [-0.00861969  0.00366574  0.00518988  0.00574194  0.00746692 -0.00616768
  0.00110561  0.00604728 -0.00284005 -0.00617352]

'machine' 和 'learning' 的余弦相似度: -0.0108


### 6.5 使用预训练词向量

在小规模数据集上从头训练 Word2Vec 往往难以获得高质量的词向量。实际工程中，通常使用在大规模语料（如 Google News, Wikipedia）上 **预训练（Pre-trained）** 好的模型。


In [10]:
import gensim.downloader as api

# 下载预训练模型 (例如 GloVe Twitter 模型，25维)
# 注意：首次运行会自动下载模型文件（约 100MB+）
try:
    print("正在加载预训练模型...")
    model_pretrained = api.load("glove-twitter-25")
    
    # 查找相似词
    print("Most similar to 'computer':")
    print(model_pretrained.most_similar("computer", topn=3))
    
    # 词汇类比
    result = model_pretrained.most_similar(positive=['woman', 'king'], negative=['man'], topn=1)
    print(f"\nWoman + King - Man = {result[0][0]}")
except Exception as e:
    print(f"加载失败: {e}")

正在加载预训练模型...
[==================================================] 100.0% 104.8/104.8MB downloaded
Most similar to 'computer':
[('camera', 0.907833456993103), ('cell', 0.891890287399292), ('server', 0.874466598033905)]

Woman + King - Man = meets


---

## 10. 三种方法的对比分析

| **特性**       | **词袋模型 (BoW)**         | **TF-IDF**                 | **Word2Vec**                     |
| :------------- | :------------------------- | :------------------------- | :------------------------------- |
| **核心思想**   | 词频统计                   | 加权词频统计               | 上下文预测（神经网络）           |
| **表示形式**   | 高维、稀疏、离散           | 高维、稀疏、离散           | **低维、稠密、连续**             |
| **维度大小**   | 词汇表大小（例如 10,000+） | 词汇表大小（例如 10,000+） | **自定义（例如 100-300）**       |
| **语义关系**   | 无法捕捉                   | 无法捕捉                   | **可以捕捉（相似度、类比）**     |
| **计算复杂度** | 低                         | 低                         | 较高（需训练）                   |
| **主要应用**   | 简单文本分类、垃圾邮件检测 | 关键词提取、搜索引擎排名   | 语义分析、推荐系统、深度学习输入 |

---

## 10. 练习与思考

1. **手动计算**：给定文档 D1="data science" 和 D2="data analysis"，请构建词汇表并计算 D1 的 TF-IDF 向量（假设简单的 TF 计数，IDF 使用 $\log(N/DF)$）。
2. **思考**：为什么 Word2Vec 的向量被称为“稠密向量”？相比于“稀疏向量”有什么好处？
3. **实践**：尝试使用 `gensim` 加载预训练好的 Word2Vec 模型（如 Google News 模型），并寻找 "Paris" - "France" + "Italy" 的结果。

---

## 10. 总结与展望

本章详细介绍了三种经典的文本向量化方法。词袋模型和 TF-IDF 虽然简单且易于产生稀疏矩阵，但仍在许多工业级应用（如简单的搜索排序）中发挥作用。Word2Vec 的出现标志着 NLP 进入了深度学习时代，它能够捕捉复杂的语义关系。

随着技术的发展，基于 **Transformer** 架构的预训练模型（如 **BERT**, **GPT**）已成为当前的主流。这些模型不仅考虑了词的上下文，还引入了**注意力机制（Self-Attention）**，解决了 Word2Vec 中“一词多义”无法区分的问题（即静态词向量问题），能够生成**动态词向量**（Contextualized Word Embeddings）。

---

## 10. 参考文献

1. [Scikit-learn Documentation](https://scikit-learn.org/stable/modules/feature_extraction.html). _Feature extraction_.
2. Mikolov, T., et al. (2013). _Efficient Estimation of Word Representations in Vector Space_. arXiv preprint arXiv:1301.3781.
3. Salton, G., & Buckley, C. (1988). _Term-weighting approaches in automatic text retrieval_. Information processing & management.